# Play with Gongfeng Provider

本文档演示如何使用 Gongfeng Runtime Provider 来管理远程沙箱环境。 Gongfeng Runtime Provider 提供了统一的生命周期管理 API，支持多种底层沙箱资源调度。

## 初始化环境

首先导入必要的模块并配置日志，以便查看 Runtime 的详细操作过程。

In [1]:
from nexus.utils.log import setup_logging
from nexus.runtime import GongfengRuntimeProvider

setup_logging()

## 创建 Runtime Provider

创建 `GongfengRuntimeProvider` 实例，指定沙箱服务的 API 地址。 这里使用测试环境的地址。

In [2]:
rp = GongfengRuntimeProvider(api_base_url='http://hyrl-sandbox.test.woa.com:8052', token='<your-token>')

## 获取 Runtime 实例

调用 `acquire()` 方法创建并启动一个新的 Runtime 实例。 需要指定：
- `name`：Runtime 实例的唯一名称
- `image`：要运行的主容器镜像

该方法会自动调用 Gongfeng API 启动沙箱，并返回一个可用的 `Runtime` 对象。

In [3]:
runtime = await rp.acquire(name='main', image='mirrors.tencent.com/hunyuan_yanguan/tbench2__mailman__ruha8w')

2026-01-13 01:23:58 | INFO | nexus.runtime.gongfeng | Starting new runtime for main with image=mirrors.tencent.com/hunyuan_yanguan/tbench2__mailman__ruha8w, runtime_image=mirrors.tencent.com/hunyuan_yanguan/nexus-runtime:latest, resource_spec=None, exposed_ports=[], working_dir=None
2026-01-13 01:23:58 | INFO | nexus.runtime.gongfeng | Gongfeng API request: http://hyrl-sandbox.test.woa.com:8052/sandbox/v3/runtime/start, payload: {'agentflow_type': 'nexus', 'image': 'mirrors.tencent.com/hunyuan_yanguan/tbench2__mailman__ruha8w', 'runtime_image': 'mirrors.tencent.com/hunyuan_yanguan/nexus-runtime:latest', 'environment': {'PYTHON_VERSION': '3.13'}}
2026-01-13 01:24:10 | INFO | nexus.runtime.gongfeng | Gongfeng API response: status=200, body={"code":0,"msg":"","data":{"runtime_id":"nexus-2293d11d28294cda8a50faca87505d93","url":"http://nexus-2293d11d28294cda8a50faca87505d93.19527.hyrl-sandbox.test.woa.com:8053","port_mappings":{}}}
2026-01-13 01:24:10 | INFO | nexus.runtime.gongfeng | Runti

## 在 Runtime 中执行命令

使用 `runtime.run_command()` 方法在沙箱环境中执行命令。 这里执行 `cat /etc/os-release` 来查看容器的操作系统信息。

命令执行结果包含标准输出（`stdout`）、标准错误（`stderr`）和退出码（`exit_code`）。

In [4]:
result = await runtime.run_command('cat /etc/os-release')

print(result.stdout)

2026-01-13 01:24:10 | INFO | nexus.core.registry | Imported 'nexus.extensions.agents.demo.tools' with 2 registrations:
2026-01-13 01:24:10 | INFO | nexus.core.registry |   - [tool] demo/think, demo/bash
2026-01-13 01:24:10 | INFO | nexus.core.registry | Imported 'nexus.extensions.agents.swe_agent.tools' with 32 registrations:
2026-01-13 01:24:10 | INFO | nexus.core.registry |   - [executor] create_browser_session, destroy_browser_session, browser_goto, browser_go_back, browser_go_forward, browser_click, browser_fill, browser_scroll, browser_press_key, browser_hover, browse [... (truncated log)] l_session, run_command_in_terminal_session, start_command_in_terminal_session, query_terminal_command_status, send_keys_to_terminal_session, capture_terminal_pane, terminate_terminal_session_processes
2026-01-13 01:24:10 | INFO | nexus.core.registry |   - [tool] swe_agent/bash, swe_agent/str_replace_editor, swe_agent/review_submit
2026-01-13 01:24:11 | INFO | nexus.core.registry | Imported 'nexu

## 释放所有 Runtime

使用 `release_all()` 方法批量停止并释放所有创建的 Runtime 实例。 这会调用 Gongfeng API 停止相应的沙箱容器，释放资源。

对于单个 Runtime 实例，也可以使用 `release(name)` 方法单独释放。

In [5]:
await rp.release_all()

2026-01-13 01:24:11 | INFO | nexus.runtime.gongfeng | Stopping all runtimes: ['nexus-2293d11d28294cda8a50faca87505d93']
2026-01-13 01:24:11 | INFO | nexus.runtime.gongfeng | Gongfeng API request: http://hyrl-sandbox.test.woa.com:8052/sandbox/v3/runtime/stop, payload: {'runtime_ids': ['nexus-2293d11d28294cda8a50faca87505d93']}
2026-01-13 01:24:11 | INFO | nexus.runtime.gongfeng | Gongfeng API response: status=200, body={"code":0,"msg":"success"}
2026-01-13 01:24:11 | INFO | nexus.runtime.gongfeng | All runtimes released


## （可选）被后端入队时的排队等待

客户端是响应式的：只要后端返回 `status=queued`（url 为空），`acquire` 就会自动轮询直到沙盒被调度。最常见的入队原因是 token 在沙盒门户配置了 `quota_full_policy=queue` 且配额已满。`queue_wait_max` 只决定客户端最多轮询多久（默认 0=无截止）。详细说明见 [Gongfeng Runtime Provider 配置文档「被后端入队时的排队等待」](../configure-runtime/#被后端入队时的排队等待)。

下面的代码假设你已经在 [混元沙盒门户](https://sbox.woa.com/?tab=resource) 把 token 改成了 `queue` 策略：

In [ ]:
from nexus.runtime.providers.gongfeng import (
    RuntimeQueueFailedError,
    RuntimeQueueWaitTimeoutError,
)

rp_queue = GongfengRuntimeProvider(
    api_base_url='http://hyrl-sandbox.test.woa.com:8052',
    token='<your-queue-policy-token>',
    queue_wait_max=600,  # 客户端最多等 10 分钟
)

try:
    runtime = await rp_queue.acquire(
        name='worker-1',
        image='mirrors.tencent.com/hunyuan_yanguan/tbench2__mailman__ruha8w',
    )
    print(await runtime.run_command('hostname'))
except RuntimeQueueWaitTimeoutError:
    print('queued too long, giving up this task')
except RuntimeQueueFailedError as e:
    print(f'backend gave up on the queued runtime: {e}')
finally:
    await rp_queue.release_all()